# Data filter
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
# Filter function
def filter_products(df, min_observations_per_month, min_months):
    # Group by 'descripcion' and get the count of observations per month
    df['month'] = df['fecha'].dt.to_period('M')  # Create a 'month' column
    monthly_counts = df[~df['precio'].isna()].groupby(['tienda', 'descripcion', 'month']).size().reset_index(name='count')
    monthly_counts['count_gt_min'] = monthly_counts['count'] > min_observations_per_month

    # Count number of months that pass the minimum criteria
    min_monthsdf = monthly_counts.groupby(['tienda', 'descripcion']).agg(months_true=('count_gt_min', 'sum')).reset_index()

    min_monthsdf["key"] = min_monthsdf["tienda"] + "_" + min_monthsdf["descripcion"]
    df["key"] = df["tienda"] + "_" + df["descripcion"]

    filtered_items = min_monthsdf[min_monthsdf['months_true']>min_months]['key']

    # Get the descriptions that meet the criteria
    valid_descriptions = filtered_items.unique()

    # Filter the original DataFrame to keep only valid descriptions
    filtered_df = df[df['key'].isin(valid_descriptions)]

    # Drop extra columns
    filtered_df = filtered_df.drop(["month", "key"], axis = 1)

    return filtered_df

In [ ]:
case = "Comparison"

In [ ]:
# load retailer data
source = wd_db + "Retailer_data.csv"
if not os.path.exists(source):
    # full dataset not downloaded: run the pipeline on the sample shipped with the repo
    source = wd_data + "sample/Retailer_data_sample.csv"
    print("Full dataset not found, using", source)
data = pd.read_csv(source)

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

# Drop previous index column
data = data.drop(columns=["Unnamed: 0"], errors="ignore")

# Drop zero prices
data = data[data["precio"] > 0]

1. Errores
2. Comparar con otra literatura
3. Pensar como ajustar las matrices pensando en la frecuencia de los datos

In [ ]:
if case == "Comparison":
    # Define your minimum and maximum dates
    max_b = max(data[data["tienda"]=="B"]["fecha"])
    min_b = min(data[data["tienda"]=="B"]["fecha"])
    max_a1 = max(data[data["tienda"]=="A1"]["fecha"])
    min_a1 = min(data[data["tienda"]=="A1"]["fecha"])
    max_a2 = max(data[data["tienda"]=="A2"]["fecha"])
    min_a2 = min(data[data["tienda"]=="A2"]["fecha"])
    max_c = max(data[data["tienda"]=="C"]["fecha"])
    min_c = min(data[data["tienda"]=="C"]["fecha"])

    min_date = max([min_b, min_a1, min_a2, min_c])
    max_date = min([max_b, max_a1, max_a2, max_c])

    # Filter the dataframe
    data = data[(data['fecha'] >= min_date) & (data['fecha'] <= max_date)]
else:
    case = "All"
    

# Define the full date range
date_range = pd.date_range(start=data['fecha'].min(), end=data['fecha'].max())

# Get unique combinations of 'descripcion' and 'tienda'
product_store_pairs = data[['descripcion', 'tienda']].drop_duplicates()

# Crear un DataFrame de la lista y agregarlo a cada fila del DataFrame original
datesdf = pd.DataFrame({'fecha': date_range}).merge(product_store_pairs, how='cross')

# Add missing values to DataFrame
data = datesdf.merge(data, on = ['tienda', 'descripcion', 'fecha'], how = 'left')

In [ ]:
# Usage of the function
min_observations_per_month, min_months = 4, 2 

data = filter_products(data, min_observations_per_month, min_months)
display(data)

In [ ]:
data.to_csv(wd_dpr + "Filter_Data_{}.csv".format(case), index = False)